# CS 6470 A6 — Neural net plus RL or GA

**50 points** · Modules 12–13 (due Module 13) · plan for about 4–6 focused hours

**Before you submit, rename this file to `CS6470_A6_LastName_FirstName.ipynb`.**
Open it in [Google Colab](https://colab.research.google.com/) (File → Upload notebook) or local Jupyter (Python 3.10+).

## How this notebook works

- It has **5 numbered tasks**, top to bottom. Cells marked **`# YOUR CODE`** and Markdown cells marked **YOUR ANSWER** are the graded work. Cells marked *Runs as-is* are provided — run them and read them.
- Every coding task ends with a **Checkpoint** describing what you should see, so you always know whether you are on track. Small differences (±0.01–0.02 on scores) are normal.
- **Hints name the tool** (class, argument, or doc link), not a line to paste. Newer to Python? Follow the Hints. Comfortable? Try each task from the description first. Experienced? There is an ungraded stretch at the end of the assignment page.
- A `# YOUR CODE` cell raises `NotImplementedError` until you replace that line — that is the notebook telling you where to work, not a bug.
- Before submitting: **Runtime → Restart and run all.** Every cell must run with no errors and no `NotImplementedError` left.
- **Part A is required. Part B: complete exactly ONE of the two options** (bandit *or* genetic algorithm). Delete the unused option's code cells before Restart and run all; only your chosen option is graded.


**Pacing:** Modules 12–13 share one calendar week. Budget 4–6 hours total: Part A early, exactly one Part B option next, then the memo. Submit by the A6 deadline in Canvas; Part B is not due a week later.


# Part A — Small neural net vs a strong baseline *(required)*

**Dataset:** sklearn's built-in `digits` — 1,797 grayscale 8×8 images of handwritten digits, 10 classes. No download; it ships with sklearn. This is the "hello world" of image classification, small enough for a laptop CPU.

The question of the week is not "can a neural net fit?" (it can) — it is **how you catch overfitting**: by always reading train and test scores side by side.


In [1]:
# Runs as-is — data + imports for Part A.
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("train:", X_train.shape, "test:", X_test.shape)


train: (1347, 64) test: (450, 64)


## Task A1 — Fit two models, report four numbers

In code:

1. `RandomForestClassifier(n_estimators=200, random_state=42)`.
2. A Pipeline: `StandardScaler()` → `MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42)` — a one-hidden-layer neural net with 64 units. (Scaling matters: neural nets train badly on raw pixel ranges.)
3. Fit both on the training split and print a small table with **train accuracy and test accuracy for both models** — four numbers.

If the MLP prints a convergence warning, raising `max_iter` to 500 is fine.

**Hints.** Tools only:
- Forest: `RandomForestClassifier(n_estimators=200, random_state=42)`.
- Net: `Pipeline` of `StandardScaler` then `MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42)`.
- Four numbers: [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) on train *and* test for both models, in one small table.

**Checkpoint.** Both models land at ≈ **1.00 train** accuracy; test is ≈ **0.97** (forest) and ≈ **0.98** (MLP). The train column being perfect is exactly the thing Task A2 asks you to interrogate.


In [2]:
# YOUR CODE — Task A1. Write one small block under each comment.

# 1. RandomForestClassifier(n_estimators=200, random_state=42)

rf = RandomForestClassifier(n_estimators=200, random_state=42)

# 2. Pipeline: StandardScaler -> MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42)

mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=42))
])

# 3. Fit both on train; print train AND test accuracy for both (four numbers)

rf.fit(X_train, y_train)
mlp_pipeline.fit(X_train, y_train)
accuracy_rf_train = accuracy_score(y_train, rf.predict(X_train))
accuracy_rf_test = accuracy_score(y_test, rf.predict(X_test))
accuracy_mlp_train = accuracy_score(y_train, mlp_pipeline.predict(X_train))
accuracy_mlp_test = accuracy_score(y_test, mlp_pipeline.predict(X_test))

results = pd.DataFrame({
    "Model": ["Random Forest", "MLP Pipeline"],
    "Train Accuracy": [accuracy_rf_train, accuracy_mlp_train],
    "Test Accuracy": [accuracy_rf_test, accuracy_mlp_test]
}) 

results

,Model,Train Accuracy,Test Accuracy
0,Random Forest,1.0,0.966667
1,MLP Pipeline,1.0,0.980000


## Task A2 — Diagnose the gap — YOUR ANSWER

**Q1.** Both models score ~1.00 on data they trained on. Which one generalizes better here, and what are the two numbers that say so?

**TODO:** your answer.

**Q2.** Name one lever that would shrink the MLP's train/test gap if it were bigger (smaller hidden layer, stronger `alpha` regularization, more data, early stopping…), and one sentence on how it works.

**TODO:** your answer.

**Q3.** Give one business reason the 2-line random forest might still be the better choice than the neural net at work, even at a slightly lower test score.

**TODO:** your answer.


# Part B — choose exactly ONE option

**Option 1 (next two tasks): epsilon-greedy bandit** — the smallest real reinforcement-learning idea: balancing *exploring* what you don't know against *exploiting* what looks best.
**Option 2 (after it): genetic algorithm feature-select** — an evolutionary optimizer, and a demonstration of how a leaky objective lies.

Newer to Python? Option 1 is ~10 short lines. Wanting depth? Option 2 connects back to the course's leakage theme.


## Option 1 · Task B1 — Implement the epsilon-greedy agent

**The setup (provided below):** three workplace "promotions" (arms). Each pull of arm `a` pays 1 with hidden probability `true_p[a]`, else 0. The agent sees only its own history: `counts[a]` pulls and `rewards[a]` total payoff so far.

Your agent, at every step `t` (2,000 steps total):

- with probability `eps`: **explore** — pick a uniformly random arm;
- otherwise: **exploit** — pick the arm with the best observed mean so far, `rewards / max(counts, 1)`;
- pull it, receive the 0/1 reward, and update that arm's `counts` and `rewards`.

Wrap it in the function `run_bandit(eps)` skeleton below so Task B2 can call it three times.

**Hints.** Tools only — name the pieces, you write the loop:
- Explore: a uniform random arm (`rng.integers`).
- Exploit: the arm with the best observed mean (`rewards / max(counts, 1)`).
- Reward draw (same pattern as the setup): compare `rng.random()` to `true_p[arm]`, then update that arm's `counts` and `rewards`.

**Checkpoint.** `run_bandit(0.1)` → the middle arm (true rate 0.50) collects the most pulls by far — around 1,600 of 2,000 with this seed.


In [ ]:
# Option 1 — environment (provided) + YOUR CODE for the agent loop.

import numpy as np

def run_bandit(eps, T=2000, seed=42):
    """Run epsilon-greedy for T steps. Return (counts, observed mean reward per arm)."""
    rng = np.random.default_rng(seed)
    true_p = np.array([0.35, 0.50, 0.42])   # hidden from a real agent; fixed here so runs are comparable
    counts = np.zeros(3)
    rewards = np.zeros(3)

    # YOUR CODE — the loop (T steps). Write one small block under each comment.
    # 1. Choose arm: explore with probability eps, else exploit best observed mean
    # 2. Draw the 0/1 reward for that arm
    # 3. Update that arm's counts and rewards
    raise NotImplementedError("Task B1: fill the blanks above, then delete this line")

    return counts.astype(int), np.round(rewards / np.maximum(counts, 1), 3)

counts, means = run_bandit(eps=0.1)
print("pulls per arm:", counts, "observed means:", means)


## Option 1 · Task B2 — Sweep epsilon and read it

In code: call `run_bandit` for `eps` in `(0.0, 0.1, 0.3)` and print pulls-per-arm and observed means for each.

**Hints.** Call `run_bandit` three times (`eps` in `(0.0, 0.1, 0.3)`). Print pulls-per-arm and observed means for each.

**Checkpoint.** `eps=0.0` locks onto **arm 0 forever** with this implementation (2,000 / 0 / 0 pulls) — it exploited its very first lucky impression and never learned the middle arm is better. `eps=0.3` finds the best arm but uses about 30% of pulls for exploration (some exploratory pulls still choose the best arm).

### YOUR ANSWER

**Q1.** In one paragraph: what went wrong at `eps=0`, what was wasted at `eps=0.3`, and why `eps=0.1` threads the needle here.

**TODO:** your answer.

**Q2.** Name one decision at your workplace (or a workplace you know) that is secretly a bandit problem, and what "reward" would be. One or two sentences.

**TODO:** your answer.


In [ ]:
# YOUR CODE — Task B2. Write one small block under each comment.

# 1. For eps in (0.0, 0.1, 0.3): call run_bandit(eps)


# 2. Print pulls-per-arm and observed means for each eps


raise NotImplementedError("Task B2: fill the blanks above, then delete this line")


## Option 2 · Task B1 — Two fitness functions: leaky vs honest

**The setup (provided below):** a small genetic algorithm evolves feature *masks* (which of the 30 breast-cancer columns a decision tree may use). Selection, crossover, and mutation are written for you — the GA prints the best fitness each generation. **You write the objective it optimizes**, twice:

1. `fitness_leaky(mask)` — fit `DecisionTreeClassifier(random_state=0)` on **all** rows and score F1 on **those same rows**.
2. `fitness_honest(mask)` — fit the same tree on the provided training split (`Xb_tr`, `yb_tr`) and score F1 on the held-out validation split (`Xb_val`, `yb_val`).

Run the GA once with each.

**Hints.** Tools only:
- Turn a boolean `mask` into column indices (`np.where`).
- Leaky: fit `DecisionTreeClassifier(random_state=0)` on **all** rows; score F1 on those same rows.
- Honest: fit on `Xb_tr` / `yb_tr`; score F1 on `Xb_val` / `yb_val`.

**Checkpoint.** The leaky run prints best fitness **1.000 in every generation** — an unpruned tree can often memorize these training rows, so the optimizer literally cannot tell candidates apart. The honest run prints varying values that end around **0.95–0.97**.

Here “honest” means the candidate is not trained on the rows used to score its fitness. Repeated selection still adapts to validation data. Neither selected fitness is an independent final test score. This exercise asks you to explain that limitation; it does not require an additional experiment.


In [ ]:
# Option 2 — GA machinery (provided) + YOUR CODE for the two fitness functions.

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_val, yb_tr, yb_val = train_test_split(
    Xb, yb, test_size=0.3, random_state=42, stratify=yb
)

def run_ga(fitness, pop=16, gens=8, seed=0):
    """Evolve feature masks; print best fitness per generation (provided — read it, don't edit it)."""
    rng = np.random.default_rng(seed)
    n_feat = Xb.shape[1]
    masks = rng.random((pop, n_feat)) > 0.5
    masks[:, 0] = True                        # never allow an empty mask
    fit = None
    for gen in range(gens):
        fit = np.array([fitness(m) for m in masks])
        print(f"gen {gen}: best fitness {fit.max():.3f} (features in best mask: {masks[np.argmax(fit)].sum()})")
        keep = masks[np.argsort(fit)[-pop // 2:]]      # selection: top half survives
        children = []
        for _ in range(pop // 2):                       # crossover + mutation
            a, b = keep[rng.integers(len(keep))], keep[rng.integers(len(keep))]
            cut = int(rng.integers(1, n_feat))
            child = np.concatenate([a[:cut], b[cut:]])
            flip = rng.random(n_feat) < 0.05
            child = np.where(flip, ~child, child)
            if not child.any():
                child[rng.integers(n_feat)] = True
            children.append(child)
        masks = np.vstack([keep, children])
    return fit.max()

# YOUR CODE — the two objectives the GA will optimize.

def fitness_leaky(mask):
    """Tree trained on ALL rows, scored on the SAME rows."""
    # 1. Columns from the mask  2. Fit on ALL rows  3. F1 on those same rows
    raise NotImplementedError("Task B1 (leaky): fill the blanks above, then delete this line")

def fitness_honest(mask):
    """Tree trained on Xb_tr/yb_tr, scored on held-out Xb_val/yb_val."""
    # 1. Columns from the mask  2. Fit on Xb_tr/yb_tr  3. F1 on Xb_val/yb_val
    raise NotImplementedError("Task B1 (honest): fill the blanks above, then delete this line")

print("--- leaky objective ---")
best_leaky = run_ga(fitness_leaky)
print("--- honest objective ---")
best_honest = run_ga(fitness_honest)
print("leaky best:", round(best_leaky, 3), "| honest best:", round(best_honest, 3))


## Option 2 · Task B2 — Explain the gap — YOUR ANSWER

**Q1.** Why does the leaky objective report perfect fitness for essentially any mask — and what does that do to the GA's ability to *select* anything?

**TODO:** your answer (2–4 sentences).

**Q2.** Why is validation fitness more informative than training fitness, yet still optimistic after searching many masks? What untouched test data would you need before making a claim about new cases?

**TODO:** your answer (1–2 sentences).

**Q3.** Name one workplace optimization (pricing rules, staffing, ad targeting, process tuning…) where the same trap — scoring candidates on the data they were fit to — could sneak in. Two or three sentences.

**TODO:** your answer.


## Final task — Memo (≤1 page) — YOUR ANSWER

The judgment question this program exists for. Replace each **TODO**.

### Would I deploy the neural net?

**TODO:** 2–3 sentences. Yes/no *for a concrete use*, plus what it needs to be safe (data volume, monitoring, fallback).

### Would I deploy the bandit / GA I studied?

**TODO:** 2–4 sentences. Same shape: a concrete use, the guardrails, the honest scope ("this was a 3-arm simulation, not production RL").

### When a simpler sklearn model is enough

**TODO:** 2–3 sentences. Your rule of thumb, grounded in this course's evidence (A3/A4 vs today).

### AI disclosure

**TODO:** Tool + what it did, or "I did not use AI."


## Before you submit

- [ ] File renamed to `CS6470_A6_LastName_FirstName.ipynb`
- [ ] **Runtime → Restart and run all** — every cell ran, no errors, no `NotImplementedError` anywhere
- [ ] All Part A + one Part B option + memo tasks done: every `# YOUR CODE` cell has your code, every **YOUR ANSWER** / **TODO** prompt has your words
- [ ] Memo written in the Markdown cells above (it stays inside this notebook — no separate PDF)
- [ ] AI-use disclosure filled in
- [ ] Exactly one Part B option completed; the unused option's code cells deleted so Restart and run all completes

Upload the `.ipynb` file to this assignment on Canvas (in Colab: File → Download → Download .ipynb).

Integrity: the notebook and memo must be yours. AI may help explain an error; it may not be the submitted analysis.
